# 🎯 DeepSeekMoE와 라우팅 메커니즘

이 노트북에서는 DeepSeek V3의 **Mixture of Experts (MoE)** 아키텍처와 혁신적인 라우팅 메커니즘을 학습합니다.

**참고 자료:**
- 논문: https://arxiv.org/pdf/2412.19437

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

print("✅ 라이브러리 로드 완료")

✅ 라이브러리 로드 완료


## 1. MoE 기본 개념

### 핵심 아이디어
**"모든 파라미터를 항상 사용할 필요가 없다!"**

- **Dense 모델**: 모든 파라미터가 모든 입력에 사용됨
- **MoE 모델**: 입력에 따라 일부 전문가만 활성화

### 결과
- 총 파라미터 수 ↑ (모델 용량 증가)
- 활성화 파라미터 수 ↓ (계산 효율성 유지)

### 핵심 수식

**1. 라우터 (Router):**
$$s = \text{softmax}(W_r \cdot h)$$

**2. Top-K 선택:**
$$\text{TopK}(s) = \{i_1, i_2, ..., i_k\}$$

**3. 가중 합:**
$$y = \sum_{i \in \text{TopK}} g_i \cdot \text{Expert}_i(h)$$

In [2]:
# 단일 Expert 구현 (SwiGLU MLP)
class Expert(nn.Module):
    def __init__(self, hidden_size, intermediate_size):
        super().__init__()
        self.gate_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.up_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)
        self.act_fn = nn.SiLU()
    
    def forward(self, x):
        return self.down_proj(self.act_fn(self.gate_proj(x)) * self.up_proj(x))

# Top-K Router 구현
class TopKRouter(nn.Module):
    def __init__(self, hidden_size, n_experts, top_k):
        super().__init__()
        self.n_experts = n_experts
        self.top_k = top_k
        self.weight = nn.Parameter(torch.empty(n_experts, hidden_size))
        nn.init.kaiming_uniform_(self.weight)
    
    def forward(self, hidden_states):
        # 라우터 로짓 계산
        router_logits = F.linear(hidden_states.float(), self.weight.float())
        
        # Top-K 선택
        top_k_weights, top_k_indices = torch.topk(router_logits, self.top_k, dim=-1)
        top_k_weights = F.softmax(top_k_weights, dim=-1)
        
        return router_logits, top_k_indices, top_k_weights

print("✅ Expert와 Router 구현 완료")

✅ Expert와 Router 구현 완료


## 📝 이해도 테스트 1: 라우팅 동작 이해

In [3]:
# 라우팅 동작 테스트
hidden_size = 256
n_experts = 8
top_k = 2

router = TopKRouter(hidden_size, n_experts, top_k)

# 입력 생성
batch_size, seq_len = 2, 4
hidden_states = torch.randn(batch_size, seq_len, hidden_size)

# 라우팅
router_logits, top_k_indices, top_k_weights = router(hidden_states.view(-1, hidden_size))

print("📊 라우팅 결과")
print("=" * 50)
print(f"입력 형태: {hidden_states.shape}")
print(f"라우터 로짓 형태: {router_logits.shape}")
print(f"Top-K 인덱스 형태: {top_k_indices.shape}")
print(f"Top-K 가중치 형태: {top_k_weights.shape}")

print("\n📋 첫 번째 토큰의 라우팅 결과:")
print(f"   선택된 전문가: {top_k_indices[0].tolist()}")
print(f"   가중치: {top_k_weights[0].tolist()}")

print("\n📊 전문가별 선택 횟수:")
for i in range(n_experts):
    count = (top_k_indices == i).sum().item()
    print(f"   전문가 {i}: {count}회")

📊 라우팅 결과
입력 형태: torch.Size([2, 4, 256])
라우터 로짓 형태: torch.Size([8, 8])
Top-K 인덱스 형태: torch.Size([8, 2])
Top-K 가중치 형태: torch.Size([8, 2])

📋 첫 번째 토큰의 라우팅 결과:
   선택된 전문가: [1, 5]
   가중치: [0.5457838177680969, 0.45421621203422546]

📊 전문가별 선택 횟수:
   전문가 0: 2회
   전문가 1: 2회
   전문가 2: 2회
   전문가 3: 2회
   전문가 4: 3회
   전문가 5: 1회
   전문가 6: 1회
   전문가 7: 3회


## 2. DeepSeekMoE 아키텍처

DeepSeek V3의 MoE는 **공유 전문가**와 **라우팅 전문가**를 결합합니다.

### 구성
- **공유 전문가 (Shared Experts)**: 모든 토큰이 사용 (2개)
- **라우팅 전문가 (Routed Experts)**: Top-K 선택된 전문가만 사용 (256개 중 8개)

### 수식
$$y = \text{FFN}_{\text{shared}}(h) + \sum_{i \in \text{TopK}} g_i \cdot \text{FFN}_i(h)$$

In [4]:
# SimpleMoE 구현
class SimpleMoE(nn.Module):
    def __init__(self, hidden_size, intermediate_size, n_routed_experts, n_shared_experts, top_k):
        super().__init__()
        self.hidden_size = hidden_size
        self.n_routed_experts = n_routed_experts
        self.n_shared_experts = n_shared_experts
        self.top_k = top_k
        
        # 라우터
        self.router = TopKRouter(hidden_size, n_routed_experts, top_k)
        
        # 라우팅 전문가들
        self.experts = nn.ModuleList([
            Expert(hidden_size, intermediate_size) for _ in range(n_routed_experts)
        ])
        
        # 공유 전문가들
        self.shared_experts = nn.ModuleList([
            Expert(hidden_size, intermediate_size) for _ in range(n_shared_experts)
        ])
    
    def forward(self, hidden_states):
        batch_size, seq_len, hidden_size = hidden_states.shape
        
        # 1. 공유 전문가 출력
        shared_output = torch.zeros_like(hidden_states)
        for expert in self.shared_experts:
            shared_output = shared_output + expert(hidden_states)
        
        # 2. 라우팅
        hidden_flat = hidden_states.view(-1, hidden_size)
        _, top_k_indices, top_k_weights = self.router(hidden_flat)
        
        # 3. 라우팅 전문가 출력 (간소화된 구현)
        routed_output = torch.zeros_like(hidden_flat)
        for i, expert in enumerate(self.experts):
            mask = (top_k_indices == i).any(dim=-1)
            if mask.any():
                expert_input = hidden_flat[mask]
                expert_output = expert(expert_input)
                # 가중치 적용 (간소화)
                weights = top_k_weights[mask][top_k_indices[mask] == i].unsqueeze(-1)
                routed_output[mask] += expert_output * weights.mean()
        
        routed_output = routed_output.view(batch_size, seq_len, hidden_size)
        
        return shared_output + routed_output, top_k_indices

print("✅ SimpleMoE 구현 완료")

✅ SimpleMoE 구현 완료


In [5]:
# MoE 동작 테스트
hidden_size = 256
intermediate_size = 512
n_routed_experts = 8
n_shared_experts = 1
top_k = 2

moe = SimpleMoE(hidden_size, intermediate_size, n_routed_experts, n_shared_experts, top_k)

batch_size, seq_len = 2, 4
hidden_states = torch.randn(batch_size, seq_len, hidden_size)

output, routing_indices = moe(hidden_states)

print("📊 MoE 동작 테스트")
print("=" * 50)
print(f"입력 형태: {hidden_states.shape}")
print(f"출력 형태: {output.shape}")
print(f"\n💡 설정:")
print(f"   라우팅 전문가 수: {n_routed_experts}")
print(f"   공유 전문가 수: {n_shared_experts}")
print(f"   토큰당 활성화 전문가: {top_k}")

# 파라미터 분석
total_params = sum(p.numel() for p in moe.parameters())
print(f"\n📊 총 파라미터 수: {total_params:,}")

📊 MoE 동작 테스트
입력 형태: torch.Size([2, 4, 256])
출력 형태: torch.Size([2, 4, 256])

💡 설정:
   라우팅 전문가 수: 8
   공유 전문가 수: 1
   토큰당 활성화 전문가: 2

📊 총 파라미터 수: 3,540,992


## 3. Auxiliary-Loss-Free Load Balancing

DeepSeek V3의 핵심 혁신 중 하나는 **auxiliary loss 없이** 로드 밸런싱을 달성하는 것입니다.

### 기존 MoE의 문제점
- 일부 전문가에 토큰이 집중되는 로드 불균형
- 해결을 위해 auxiliary loss 추가 → 학습 복잡도 증가

### DeepSeek V3의 해결책: Bias-based Load Balancing
$$s'_i = s_i + b_i$$

- 과부하된 전문가의 bias를 낮춤
- 덜 사용되는 전문가의 bias를 높임
- Auxiliary loss 없이 자연스러운 밸런싱!

In [6]:
# 로드 밸런싱 시뮬레이션
def simulate_routing(n_experts=8, n_tokens=1000, top_k=2, bias=None):
    # 랜덤 라우터 로짓 생성 (일부 전문가가 더 인기 있음)
    popularity = torch.randn(n_experts) * 2  # 전문가별 인기도
    router_logits = torch.randn(n_tokens, n_experts) + popularity
    
    # Bias 적용
    if bias is not None:
        router_logits = router_logits + bias
    
    # Top-K 선택
    _, indices = torch.topk(router_logits, top_k, dim=-1)
    
    # 전문가별 로드 계산
    loads = torch.zeros(n_experts)
    for i in range(n_experts):
        loads[i] = (indices == i).sum().item()
    
    return loads

# Bias 없이
loads_no_bias = simulate_routing()
print("📊 Bias 없는 라우팅 (불균형)")
print(f"전문가별 로드: {loads_no_bias.tolist()}")
print(f"표준편차: {loads_no_bias.std().item():.2f}")

# Bias 적용 (덜 사용된 전문가에 높은 bias)
bias = -loads_no_bias / loads_no_bias.max() * 2  # 인기 있는 전문가에 패널티
loads_with_bias = simulate_routing(bias=bias)
print("\n📊 Bias 적용 후 (균형)")
print(f"전문가별 로드: {loads_with_bias.tolist()}")
print(f"표준편차: {loads_with_bias.std().item():.2f}")

print("\n💡 Bias를 통해 로드 밸런싱이 개선됨!")

📊 Bias 없는 라우팅 (불균형)
전문가별 로드: [3.0, 51.0, 292.0, 0.0, 308.0, 25.0, 371.0, 950.0]
표준편차: 321.49

📊 Bias 적용 후 (균형)
전문가별 로드: [737.0, 16.0, 108.0, 327.0, 0.0, 812.0, 0.0, 0.0]
표준편차: 342.55

💡 Bias를 통해 로드 밸런싱이 개선됨!


## 3.1 HuggingFace DeepSeek V3 구현 스타일

실제 HuggingFace의 DeepSeek V3 구현을 참고하여 Load Balancing을 적용합니다.

**참고 코드:** https://github.com/huggingface/transformers/blob/main/src/transformers/models/deepseek_v3/modeling_deepseek_v3.py

### HuggingFace 구현의 핵심 특징
1. **`e_score_correction_bias`**: `register_buffer`로 관리되는 bias
2. **Sigmoid 기반 라우팅**: softmax 대신 sigmoid 사용
3. **Group-based Top-K**: 전문가를 그룹으로 나누어 선택
4. **Bias 업데이트는 학습 스크립트에서 별도 처리** (추론 코드에는 없음)


In [7]:
# HuggingFace 스타일 Router (DeepseekV3TopkRouter)
class DeepseekV3TopkRouter(nn.Module):
    """
    HuggingFace DeepSeek V3의 실제 Router 구현
    
    핵심:
    - e_score_correction_bias: register_buffer로 관리 (gradient 없음)
    - 추론 시에는 bias 업데이트 없음
    - 학습 시 bias 업데이트는 별도 함수로 처리
    """
    def __init__(self, hidden_size, n_routed_experts):
        super().__init__()
        self.n_routed_experts = n_routed_experts
        self.hidden_size = hidden_size
        
        # 라우터 가중치 (학습됨)
        self.weight = nn.Parameter(torch.empty((n_routed_experts, hidden_size)))
        nn.init.kaiming_uniform_(self.weight)
        
        # 💡 핵심: e_score_correction_bias (HuggingFace 실제 코드와 동일)
        # - register_buffer: gradient 없이 모델과 함께 저장/로드
        # - 학습 시 별도로 업데이트
        self.register_buffer("e_score_correction_bias", torch.zeros(n_routed_experts))
    
    def forward(self, hidden_states):
        """
        Router forward (HuggingFace 스타일)
        
        Returns:
            router_logits: [n_tokens, n_experts] - bias 적용 전 로짓
        """
        hidden_states = hidden_states.view(-1, self.hidden_size)
        # float32로 계산 (수치 안정성)
        router_logits = F.linear(hidden_states.type(torch.float32), self.weight.type(torch.float32))
        return router_logits

print("✅ DeepseekV3TopkRouter 구현 완료 (HuggingFace 스타일)")


✅ DeepseekV3TopkRouter 구현 완료 (HuggingFace 스타일)


In [8]:
# HuggingFace 스타일 MoE (DeepseekV3MoE)
class DeepseekV3MoE(nn.Module):
    """
    HuggingFace DeepSeek V3의 실제 MoE 구현
    
    핵심 특징:
    - Sigmoid 기반 라우팅 (softmax 대신)
    - e_score_correction_bias로 로드 밸런싱
    - Group-based Top-K 선택
    - 공유 전문가 + 라우팅 전문가
    """
    def __init__(
        self,
        hidden_size,
        intermediate_size,
        n_routed_experts,
        n_shared_experts,
        top_k,
        n_group=1,           # 전문가 그룹 수
        topk_group=1,        # 선택할 그룹 수
        norm_topk_prob=True,
        routed_scaling_factor=1.0,
    ):
        super().__init__()
        self.hidden_size = hidden_size
        self.n_routed_experts = n_routed_experts
        self.n_shared_experts = n_shared_experts
        self.top_k = top_k
        self.n_group = n_group
        self.topk_group = topk_group
        self.norm_topk_prob = norm_topk_prob
        self.routed_scaling_factor = routed_scaling_factor
        
        # Router (HuggingFace 스타일)
        self.gate = DeepseekV3TopkRouter(hidden_size, n_routed_experts)
        
        # 라우팅 전문가들
        self.experts = nn.ModuleList([
            Expert(hidden_size, intermediate_size) for _ in range(n_routed_experts)
        ])
        
        # 공유 전문가 (하나의 큰 MLP로 구현)
        self.shared_experts = Expert(hidden_size, intermediate_size * n_shared_experts)
    
    def route_tokens_to_experts(self, router_logits):
        """
        HuggingFace의 route_tokens_to_experts 메서드와 동일한 구현
        
        핵심: sigmoid + e_score_correction_bias
        """
        # 1. Sigmoid 적용 (softmax 대신!)
        router_logits = router_logits.sigmoid()
        
        # 2. 💡 핵심: Bias 적용 (Load Balancing)
        router_logits_for_choice = router_logits + self.gate.e_score_correction_bias
        
        # 3. Group-based Top-K (간소화 버전)
        if self.n_group > 1:
            # 그룹별로 상위 2개 선택 후 합산
            group_scores = (
                router_logits_for_choice.view(-1, self.n_group, self.n_routed_experts // self.n_group)
                .topk(2, dim=-1)[0]
                .sum(dim=-1)
            )
            group_idx = torch.topk(group_scores, k=self.topk_group, dim=-1, sorted=False)[1]
            group_mask = torch.zeros_like(group_scores)
            group_mask.scatter_(1, group_idx, 1)
            score_mask = (
                group_mask.unsqueeze(-1)
                .expand(-1, self.n_group, self.n_routed_experts // self.n_group)
                .reshape(-1, self.n_routed_experts)
            )
            scores_for_choice = router_logits_for_choice.masked_fill(~score_mask.bool(), 0.0)
        else:
            scores_for_choice = router_logits_for_choice
        
        # 4. Top-K 선택
        topk_indices = torch.topk(scores_for_choice, k=self.top_k, dim=-1, sorted=False)[1]
        
        # 5. 원래 router_logits에서 가중치 추출 (bias 적용 전!)
        topk_weights = router_logits.gather(1, topk_indices)
        
        # 6. 정규화 (선택사항)
        if self.norm_topk_prob:
            denominator = topk_weights.sum(dim=-1, keepdim=True) + 1e-20
            topk_weights = topk_weights / denominator
        
        # 7. Scaling factor 적용
        topk_weights = topk_weights * self.routed_scaling_factor
        
        return topk_indices, topk_weights
    
    def forward(self, hidden_states):
        batch_size, seq_len, _ = hidden_states.shape
        residuals = hidden_states
        
        # 1. Router 로짓 계산
        router_logits = self.gate(hidden_states)
        
        # 2. 토큰-전문가 라우팅 (bias 적용됨!)
        topk_indices, topk_weights = self.route_tokens_to_experts(router_logits)
        
        # 3. 라우팅 전문가 출력
        hidden_flat = hidden_states.view(-1, self.hidden_size)
        routed_output = torch.zeros_like(hidden_flat)
        
        for i, expert in enumerate(self.experts):
            mask = (topk_indices == i).any(dim=-1)
            if mask.any():
                expert_input = hidden_flat[mask]
                expert_output = expert(expert_input)
                
                # 가중치 적용
                weights_for_expert = topk_weights[mask]
                indices_for_expert = topk_indices[mask]
                weight_mask = (indices_for_expert == i)
                weights = (weights_for_expert * weight_mask).sum(dim=-1, keepdim=True)
                
                routed_output[mask] += expert_output * weights
        
        routed_output = routed_output.view(batch_size, seq_len, self.hidden_size)
        
        # 4. 공유 전문가 출력 + 라우팅 전문가 출력
        output = routed_output + self.shared_experts(residuals)
        
        return output, topk_indices

print("✅ DeepseekV3MoE 구현 완료 (HuggingFace 스타일)")


✅ DeepseekV3MoE 구현 완료 (HuggingFace 스타일)


In [9]:
# MoE 동작 테스트 (HuggingFace 스타일)
print("🧪 DeepseekV3MoE 동작 테스트")
print("=" * 60)

# 설정
hidden_size = 256
intermediate_size = 512
n_routed_experts = 8
n_shared_experts = 1
top_k = 2

moe = DeepseekV3MoE(
    hidden_size=hidden_size,
    intermediate_size=intermediate_size,
    n_routed_experts=n_routed_experts,
    n_shared_experts=n_shared_experts,
    top_k=top_k,
)

# 입력 생성
batch_size, seq_len = 2, 4
hidden_states = torch.randn(batch_size, seq_len, hidden_size)

# Forward
output, topk_indices = moe(hidden_states)

print(f"입력 형태: {hidden_states.shape}")
print(f"출력 형태: {output.shape}")
print(f"Top-K 인덱스 형태: {topk_indices.shape}")

print("\n📊 전문가별 선택 횟수:")
for i in range(n_routed_experts):
    count = (topk_indices == i).sum().item()
    print(f"   전문가 {i}: {count}회")

print(f"\n💡 e_score_correction_bias (초기값): {moe.gate.e_score_correction_bias.tolist()}")


🧪 DeepseekV3MoE 동작 테스트
입력 형태: torch.Size([2, 4, 256])
출력 형태: torch.Size([2, 4, 256])
Top-K 인덱스 형태: torch.Size([8, 2])

📊 전문가별 선택 횟수:
   전문가 0: 2회
   전문가 1: 2회
   전문가 2: 2회
   전문가 3: 2회
   전문가 4: 2회
   전문가 5: 2회
   전문가 6: 2회
   전문가 7: 2회

💡 e_score_correction_bias (초기값): [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


In [10]:
# 학습 시 Bias 업데이트 함수 (별도 구현 필요)
def update_e_score_correction_bias(
    moe_module: DeepseekV3MoE,
    topk_indices: torch.Tensor,
    n_tokens: int,
    bias_update_speed: float = 0.001,
):
    """
    학습 스크립트에서 호출하는 bias 업데이트 함수
    
    HuggingFace 추론 코드에는 없지만, 학습 시 필요한 로직
    
    Args:
        moe_module: MoE 모듈
        topk_indices: 선택된 전문가 인덱스 [n_tokens, top_k]
        n_tokens: 토큰 수
        bias_update_speed: bias 업데이트 속도
    """
    with torch.no_grad():
        n_experts = moe_module.n_routed_experts
        top_k = moe_module.top_k
        target_load = top_k / n_experts  # 균등 분배 목표
        
        # 전문가별 사용 비율 계산
        expert_counts = torch.zeros(n_experts, device=topk_indices.device)
        for i in range(n_experts):
            expert_counts[i] = (topk_indices == i).sum().float()
        
        current_load = expert_counts / (n_tokens * top_k)
        
        # Bias 업데이트: 목표와의 차이에 비례
        # 많이 사용됨 → bias 감소, 적게 사용됨 → bias 증가
        load_diff = target_load - current_load
        moe_module.gate.e_score_correction_bias += bias_update_speed * load_diff

print("✅ update_e_score_correction_bias 함수 정의 완료")


✅ update_e_score_correction_bias 함수 정의 완료


In [11]:
# 실제 학습 루프 예시 (HuggingFace 스타일)
print("🎓 학습 루프에서 Bias 업데이트 예시")
print("=" * 60)

# MoE 모델 생성
moe_model = DeepseekV3MoE(
    hidden_size=256,
    intermediate_size=512,
    n_routed_experts=8,
    n_shared_experts=1,
    top_k=2,
)

# 옵티마이저 설정
optimizer = torch.optim.Adam(moe_model.parameters(), lr=1e-4)

print("\n📊 학습 루프 시뮬레이션:")
print("-" * 60)

moe_model.train()
for epoch in range(3):
    for step in range(5):
        # 가상 데이터
        batch = torch.randn(4, 8, 256)  # [batch, seq_len, hidden]
        target = torch.randn(4, 8, 256)
        
        # Forward
        output, topk_indices = moe_model(batch)
        
        # Loss 계산
        loss = F.mse_loss(output, target)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # 💡 핵심: Bias 업데이트 (학습 스크립트에서 별도 호출!)
        n_tokens = batch.shape[0] * batch.shape[1]
        update_e_score_correction_bias(
            moe_model, 
            topk_indices, 
            n_tokens,
            bias_update_speed=0.01  # 시뮬레이션을 위해 빠르게 설정
        )
    
    # 에폭 끝에서 상태 확인
    print(f"\nEpoch {epoch}:")
    print(f"  Loss: {loss.item():.4f}")
    bias_values = moe_model.gate.e_score_correction_bias.tolist()
    print(f"  e_score_correction_bias: {[f'{b:.4f}' for b in bias_values]}")

print("\n" + "=" * 60)
print("💡 HuggingFace 스타일 핵심 포인트:")
print("   1. e_score_correction_bias는 register_buffer로 저장")
print("   2. 추론 코드(HuggingFace)에는 bias 업데이트 로직이 없음")
print("   3. 학습 스크립트에서 별도로 update_e_score_correction_bias() 호출")
print("   4. Sigmoid 기반 라우팅 사용 (softmax 대신)")


🎓 학습 루프에서 Bias 업데이트 예시

📊 학습 루프 시뮬레이션:
------------------------------------------------------------

Epoch 0:
  Loss: 1.0171
  e_score_correction_bias: ['0.0055', '0.0052', '0.0070', '0.0070', '0.0052', '0.0067', '0.0061', '0.0073']

Epoch 1:
  Loss: 1.0024
  e_score_correction_bias: ['0.0108', '0.0123', '0.0130', '0.0125', '0.0120', '0.0122', '0.0123', '0.0148']

Epoch 2:
  Loss: 0.9968
  e_score_correction_bias: ['0.0170', '0.0177', '0.0195', '0.0200', '0.0177', '0.0191', '0.0198', '0.0192']

💡 HuggingFace 스타일 핵심 포인트:
   1. e_score_correction_bias는 register_buffer로 저장
   2. 추론 코드(HuggingFace)에는 bias 업데이트 로직이 없음
   3. 학습 스크립트에서 별도로 update_e_score_correction_bias() 호출
   4. Sigmoid 기반 라우팅 사용 (softmax 대신)


## 📝 이해도 테스트 2: MoE 효율성 계산

In [12]:
# DeepSeek V3 스케일에서 효율성 계산
hidden_size = 7168
intermediate_size = 18432
moe_intermediate_size = 2048
n_routed_experts = 256
n_shared_experts = 2
top_k = 8

# Dense MLP 파라미터
dense_params = 3 * hidden_size * intermediate_size

# MoE 파라미터
expert_params = 3 * hidden_size * moe_intermediate_size
routed_params = n_routed_experts * expert_params
shared_params = n_shared_experts * expert_params
router_params = n_routed_experts * hidden_size
total_moe_params = routed_params + shared_params + router_params

# 활성화 파라미터
active_params = (top_k + n_shared_experts) * expert_params

print("📊 MoE 효율성 분석 (DeepSeek V3 스케일)")
print("=" * 60)
print(f"Dense MLP 파라미터:        {dense_params:>15,}")
print(f"MoE 총 파라미터:           {total_moe_params:>15,}")
print(f"MoE 활성화 파라미터:       {active_params:>15,}")
print()
print(f"📈 파라미터 증가율: {total_moe_params/dense_params:.1f}x")
print(f"📈 활성화 비율: {active_params/total_moe_params*100:.1f}%")
print(f"📈 Dense 대비 계산량: {active_params/dense_params*100:.1f}%")

print("\n💡 핵심: 파라미터는 많지만 계산량은 적다!")

📊 MoE 효율성 분석 (DeepSeek V3 스케일)
Dense MLP 파라미터:            396,361,728
MoE 총 파라미터:            11,364,204,544
MoE 활성화 파라미터:           440,401,920

📈 파라미터 증가율: 28.7x
📈 활성화 비율: 3.9%
📈 Dense 대비 계산량: 111.1%

💡 핵심: 파라미터는 많지만 계산량은 적다!


## 4. 요약 퀴즈

In [13]:
quiz = {
    "Q1: MoE는 모든 전문가를 항상 사용한다": None,
    "Q2: DeepSeek V3는 256개 전문가 중 8개를 활성화한다": None,
    "Q3: 공유 전문가는 라우팅 없이 모든 토큰에 사용된다": None,
    "Q4: Auxiliary loss가 로드 밸런싱에 필수적이다": None,
}

# 여기에 답을 입력하세요 (True or False)

In [23]:
answers = {
    "Q1: MoE는 모든 전문가를 항상 사용한다": False,
    "Q2: DeepSeek V3는 256개 전문가 중 8개를 활성화한다": True,
    "Q3: 공유 전문가는 라우팅 없이 모든 토큰에 사용된다": True,
    "Q4: Auxiliary loss가 로드 밸런싱에 필수적이다": False,  # DeepSeek V3는 bias로 해결
}

print("📋 퀴즈 정답")
print("=" * 50)
for q, a in answers.items():
    user_ans = quiz.get(q)
    status = "✅" if user_ans == a else "❌"
    print(f"{status} {q}")
    print(f"   정답: {a}\n")

📋 퀴즈 정답
❌ Q1: MoE는 모든 전문가를 항상 사용한다
   정답: False

❌ Q2: DeepSeek V3는 256개 전문가 중 8개를 활성화한다
   정답: True

❌ Q3: 공유 전문가는 라우팅 없이 모든 토큰에 사용된다
   정답: True

❌ Q4: Auxiliary loss가 로드 밸런싱에 필수적이다
   정답: False

